# Lab 1: Structured outputs

This lab makes a model return data in a fixed shape that a program can trust. The example is a company, TechCorp, that reads social media posts about its app and its website.

- Part 1: define the structure, call the model with it, and validate replies.
- Part 2: check the stop reason, retry with the error list, see what retries cost, and compare with structured output.
- Part 3 (optional): validate string formats.

Complete each `# TODO` and run the check cell at the end of each part. Part 3 is optional.

## Load the model

This cell installs `anthropic`, sets `MODEL` to `claude-sonnet-5-5`, reads the key from the Colab secret `AZURE_API_KEY`, and creates `client` for the Foundry endpoint. Add the secret with the key icon in the sidebar and turn on Notebook access before you run this cell; never paste the key into a cell. Expect no output, or a note about restarting the kernel.

In [ ]:
%pip install -q anthropic
from anthropic import Anthropic
from google.colab import userdata

endpoint = "https://evergreen-ai-dev-resource.services.ai.azure.com/anthropic"
MODEL = "claude-sonnet-5-5"  # the deployment name
api_key = userdata.get("AZURE_API_KEY")  # Colab Secrets (key icon, left sidebar)

client = Anthropic(base_url=endpoint, api_key=api_key)

## Part 1: Define the structure (required)

### A small class: Author

A schema is a Pydantic class. `Author` has a text field, a whole-number field, and an optional text field. This cell defines the class. Expect no output.

In [ ]:
from typing import Optional
from pydantic import BaseModel

class Author(BaseModel):
    handle: str
    followers: int
    bio: Optional[str] = None

This cell sends a request with `Author` as the schema (`output_format`) and prints the parsed object. Expect a made-up author that changes on every run.

In [ ]:
r = client.messages.parse(
    model=MODEL, max_tokens=1000, output_format=Author,
    system="You are a helpful assistant.",
    messages=[{"role": "user", "content": "Make up a social media author who posts about our app."}],
)
print(r.parsed_output)

### Define the structure

**Exercise 1.** Complete the class `Review`. The field `sentiment` must accept only positive, negative, and neutral, and the class must reject extra fields. The other two fields are given. The check at the end of Part 1 expects any other sentiment and any extra field to be rejected. Expect no output.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, ValidationError

# TODO 1: Restrict sentiment to positive, negative, and neutral; reject extra fields.
# Hint 1: Literal["a", "b"] lists the allowed values of a field.
# Hint 2: Inside the class, set model_config = ConfigDict(...); the setting is called extra and takes allow, ignore, or forbid.
class Review(BaseModel):
    # YOUR CODE HERE (add the configuration, then sentiment)
    needs_reply: bool
    reply: str

### Example posts

This cell defines `POST` and `posts`, a list of three short social media posts: praise for the app, a complaint about the website, and a neutral remark. Expect no output.

In [ ]:
POST = "Your app is amazing! The new design is perfect."

posts = [
    POST,
    "The website is down again, please fix!",
    "Just got the app today.",
]

### Analyze a post

This cell defines `REVIEW_PROMPT` and `analyze_post`, which returns one structured result for a post in the requested tone. Requires Exercise 1. Expect no output.

In [ ]:
# The prompt says what to extract; the tone is added when the function is called.
REVIEW_PROMPT = ("You handle social media posts about TechCorp, a company with an app and a website. "
                 "Reply to the author in two sentences, set the sentiment to positive, negative, "
                 "or neutral, and set needs_reply to true when the post is a complaint, "
                 "a question, or praise that deserves a thank-you.")

def analyze_post(post, tone="friendly"):
    """Returns a Review object for one post; the reply is written in the given tone."""
    r = client.messages.parse(
        model=MODEL, max_tokens=1000, output_format=Review,
        system=f"{REVIEW_PROMPT} Write the reply in a {tone} tone.",
        messages=[{"role": "user", "content": post}],
    )
    return r.parsed_output

This cell analyzes `POST` and prints the object and the route. Requires Exercise 1. Expect `sentiment` positive, `needs_reply` set to true, and the route `send reply`.

In [ ]:
# The class is converted to a JSON Schema that constrains the reply.
review = analyze_post(POST)
print(review)
print("Route:", "send reply" if review.needs_reply else "no reply")

### Change the tone

This cell analyzes `POST` again with `tone="formal"` and prints only the reply. Expect a more formal reply than in the previous cell.

In [ ]:
formal = analyze_post(POST, tone="formal")
print(formal.reply)

This cell prints `formal` as JSON with `model_dump_json`. Requires Exercise 1. Expect the three fields `sentiment`, `needs_reply`, and `reply`. The keys and the types stay the same whatever the tone.

In [ ]:
print(formal.model_dump_json(indent=2))

### Print the JSON Schema

This cell prints the JSON Schema that Pydantic generates from the class. Requires Exercise 1. Expect three `sentiment` values and `additionalProperties` set to false.

In [ ]:
import json

print(json.dumps(Review.model_json_schema(), indent=2))

### Validate a reply

**Exercise 2.** Requires Exercise 1. Inside the `try` block, validate `raw` against `Review` and store the result in `results[name]`. The `except` block already stores `rejected` for an invalid reply. Expect the line `good ->` followed by a `Review` object, and the line `bad -> rejected`.

In [ ]:
GOOD = ('{"sentiment": "positive", "needs_reply": true, '
        '"reply": "Thank you for the kind words! We are glad you like the new design."}')
BAD = GOOD.replace('"positive"', '"great"')  # great is not an allowed sentiment

# TODO 2: Validate raw against Review and store the outcome in results[name].
# Hint 1: Pydantic can validate a JSON string against a class in one call.
# Hint 2: Use the class method whose name starts with model_validate and mentions JSON.
results = {"good": None, "bad": None}
for name, raw in [("good", GOOD), ("bad", BAD)]:
    try:
        # YOUR CODE HERE
        pass
    except ValidationError:
        results[name] = "rejected"
    print(name, "->", results[name])

This cell prints the full validation error for `BAD`. Expect a short multi-line message with the field `sentiment`, the rule, the rejected value `great`, and a link to the Pydantic documentation. If it says `BAD passed validation`, complete Exercise 1 and run it again.

In [ ]:
try:
    Review.model_validate_json(BAD)
except ValidationError as e:
    print(e)
else:
    print("BAD passed validation. Complete Exercise 1, then run this cell again.")

### Report the errors

**Exercise 3.** Write `error_list` to return one line per problem in a `ValidationError`, in the form `field: message`, joined with newline characters into one string. The field is the first item of the problem location. When the location is empty, such as for text that is not valid JSON, use the field `reply`, the name of the whole reply. Expect no output.

In [ ]:
# TODO 3: Return one "field: message" line per problem.
# Hint 1: A validation error holds a list of problems, each with a location and a message.
# Hint 2: The method errors() returns the problems as dictionaries with the keys loc and msg; loc is empty for text that is not valid JSON.
def error_list(e):
    """Returns one 'field: message' line per problem in a ValidationError."""
    # YOUR CODE HERE
    return ""

This cell prints the error list for `BAD`. Expect one line that names `sentiment`.

In [ ]:
try:
    Review.model_validate_json(BAD)
except ValidationError as e:
    print(error_list(e))
else:
    print("BAD passed validation. Complete Exercise 1, then run this cell again.")

### Use the structure in code

This cell loops over `posts`, calls `analyze_post` for each one, and prints the post, the sentiment, `needs_reply`, and the route. Each result is stored with its post text in `rows`. Requires Exercise 1. Expect three short blocks of output, one per post.

In [ ]:
rows = []
if "sentiment" not in Review.model_fields:
    print("Complete Exercise 1 and run its cell first, then run this cell again.")
else:
    for post in posts:
        result = analyze_post(post)
        route = "send reply" if result.needs_reply else "no reply"
        print("Post:", post)
        print("  sentiment:", result.sentiment, "| needs_reply:", result.needs_reply, "| route:", route)
        row = result.model_dump()
        row["post"] = post
        rows.append(row)

This cell installs pandas if it is missing, prints the number of posts that need a reply, and shows `rows` as a table. Requires the previous cell to have run. Expect the count line and one row per post with the columns `sentiment`, `needs_reply`, `reply`, and `post`.

In [ ]:
%pip install -q pandas
import pandas as pd

if rows:
    df = pd.DataFrame(rows)
    print("Posts that need a reply:", int(df["needs_reply"].sum()))
    display(df)
else:
    print("No rows yet. Complete Exercise 1 and run the previous cell first.")

### Check your work

This cell defines `check`, `Probe`, `accepts`, and `fields`, which test the structure and `error_list` without a model call. Expect no output.

In [ ]:
# Helpers for the check cells; no model call is made.
def check(n, test, expected):
    """Runs one offline test and prints a message when it fails."""
    try:
        passed, why = bool(test()), ""
    except Exception as e:
        passed, why = False, f" ({type(e).__name__})"
    if not passed:
        print(f"Exercise {n} is incomplete. Expected: {expected}{why}")
    return passed

class Probe(BaseModel):  # minimal class used to test error_list
    a: int
    b: int

def accepts(**kw):
    """Returns True when Review accepts the given field values."""
    try:
        Review(**(dict(sentiment="positive", needs_reply=False, reply="x") | kw))
        return True
    except Exception:
        return False

def fields(raw):
    """Returns the field names that error_list reports for an invalid Probe reply."""
    try:
        Probe.model_validate_json(raw)
    except ValidationError as e:
        return [x.split(": ", 1)[0] if ": " in x else None
                for x in error_list(e).splitlines()]

This cell checks Exercises 1 to 3 offline and prints each incomplete exercise or a pass message.

In [ ]:
# Check for Part 1; no model call is made.
done = [
    check(1, lambda: all(accepts(sentiment=s) for s in ("positive", "negative", "neutral")),
          "sentiment accepts positive, negative, and neutral."),
    check(1, lambda: not accepts(sentiment="great"),
          "sentiment rejects any other value, such as great."),
    check(1, lambda: not accepts(color="red"),
          "Review rejects extra fields."),
    check(2, lambda: type(results.get("good")).__name__ == "Review" and results.get("bad") == "rejected",
          "GOOD yields a Review object and BAD yields \"rejected\" (BAD requires Exercise 1; run Exercise 2 again after changing Review)."),
    check(3, lambda: fields('{"a": "x", "b": "y"}') == ["a", "b"],
          "error_list returns one \"field: message\" line per problem."),
    check(3, lambda: fields("not json") == ["reply"],
          "error_list uses the field \"reply\" for a problem with no location, such as text that is not valid JSON."),
]
print("Part 1: all checks passed." if all(done)
      else "Complete the open exercises, then run this cell again.")

## Part 2: Validate and retry (required)

This part adds a validation loop: stop reason check, validation, retry with the error list, and escalation. It uses `client`, `POST`, `Review`, and `error_list` from Part 1, so run the cells in order.

This cell requests JSON through the prompt alone. The prompt lists the sentiment labels good, bad, and ok, which the structure does not accept. Expect `end_turn`, the raw JSON with a label such as `good`, and then a rejected `sentiment` error. If the cell says `accepted`, complete Exercise 1 and run it again.

In [ ]:
# The prompt lists sentiment labels that Review does not allow, so validation fails.
SYSTEM = ("Reply with one JSON object only, no code fences. "
          "Keys: sentiment (good, bad or ok), needs_reply (true or false), "
          "and reply (two sentences).")
r = client.messages.create(model=MODEL, max_tokens=1000, system=SYSTEM,
                           messages=[{"role": "user", "content": POST}])
print(r.stop_reason)
raw = "".join(b.text for b in r.content if b.type == "text").strip()  # skips thinking blocks
print(raw)
try:
    print("accepted:", Review.model_validate_json(raw))
except ValidationError as e:
    print("rejected:", error_list(e), sep="\n")

### Check the stop reason

**Exercise 4.** Write `can_parse` to return `True` only for the stop reason of a reply that finished normally, and `False` for every other stop reason, including `max_tokens`, `refusal`, `tool_use`, and `stop_sequence`. A reply that is cut off or refused may not be valid JSON. Expect no output.

In [ ]:
# TODO 4: Return True only when the stop reason shows a complete reply.
# Hint 1: Compare the stop reason with the value for a reply that finished normally.
# Hint 2: A normal finish has one stop reason; every other reply has a different one.
def can_parse(stop_reason):
    """Returns True when the stop reason shows a complete reply."""
    # YOUR CODE HERE
    return False

### Ask again with the errors

**Exercise 5.** Write `retry_message` to return the user message sent after a failed validation. The message must contain the error text unchanged and ask for a JSON object only. The check looks for the error text and the word JSON. Expect no output.

In [ ]:
# TODO 5: Return a message that includes the errors and asks for JSON only.
# Hint 1: The message states what was wrong and that the reply must be JSON only.
# Hint 2: An f-string inserts the error text.
def retry_message(errors):
    """Returns the user message that asks the model to correct its reply."""
    # YOUR CODE HERE
    return "Try again."

### Put it together

This cell defines `extract`, which checks the stop reason, validates the reply, retries with the error list, and escalates at the retry limit. It makes at most `max_retries + 1` model calls, three by default. It returns a typed object or a string that starts with `escalate:`, prints an `Attempt` line for each rejected reply and for the accepted one, and records the token usage of every model call in `usage`. It uses `SYSTEM` from the prompt-only cell and requires Exercises 1, 3, 4, and 5. Expect no output.

In [ ]:
usage = []  # one entry per model call

def extract(post, max_retries=2, max_tokens=1000, system=None):
    """Requests JSON, validates it, retries with the error list, then escalates."""
    messages = [{"role": "user", "content": post}]
    for attempt in range(max_retries + 1):
        r = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                   system=system or SYSTEM, messages=messages)
        usage.append(r.usage)
        if not can_parse(r.stop_reason):
            return "escalate: stop reason " + r.stop_reason
        raw = "".join(b.text for b in r.content if b.type == "text").strip()  # skips thinking blocks
        try:
            result = Review.model_validate_json(raw)
            print(f"Attempt {attempt} accepted.")
            return result
        except ValidationError as e:
            print(f"Attempt {attempt} rejected. Errors:\n{error_list(e)}")
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": retry_message(error_list(e))})
    return f"escalate: still invalid after {max_retries} retries"

### Run the retry loop

This cell runs `extract` on `POST`. Requires Exercises 1, 3, 4, and 5. Expect an `Attempt 0 rejected` line with the error, then `Attempt 1 accepted.`, then the valid result. To explore, call `extract("Just got the app today.")` and compare the attempts.

In [ ]:
print(extract(POST))

### When retries fail

This cell runs `extract` with `max_retries=0` and with `max_tokens=12`. Requires Exercises 1, 3, 4, and 5. Expect `Attempt 0 rejected` and `escalate: still invalid after 0 retries` for the first call, and `escalate: stop reason max_tokens` for the second.

In [ ]:
print(extract(POST, max_retries=0))  # no retries allowed
print(extract(POST, max_tokens=12))  # reply cut off at 12 tokens

### What retries cost

This cell clears `usage`, runs `extract` on `POST` again, and prints the result, the number of model calls, and the input and output tokens of each call. Requires Exercises 1, 3, 4, and 5. Expect two calls, with the second call using more input tokens than the first because it resends the first reply and the error list.

In [ ]:
usage.clear()
print(extract(POST))
print("Model calls:  ", len(usage))
print("Input tokens: ", [u.input_tokens for u in usage])
print("Output tokens:", [u.output_tokens for u in usage])

### When every retry fails

This cell clears `usage` and runs `extract` with a prompt that always sets the sentiment to `great`, which the structure rejects. Requires Exercises 1, 3, 4, and 5. Expect three `Attempt` lines, three model calls with growing input tokens, and an `escalate: still invalid after 2 retries` message. If the model corrects itself, you will see fewer calls, which is also a valid outcome.

In [ ]:
ALWAYS_GREAT = ("Reply with one JSON object only, no code fences. Keys: sentiment, needs_reply, reply. "
                "Always set sentiment to great. Never use any other sentiment, even if you are asked to correct it.")

usage.clear()
print(extract(POST, system=ALWAYS_GREAT))
print("Model calls:  ", len(usage))
print("Input tokens: ", [u.input_tokens for u in usage])
print("Output tokens:", [u.output_tokens for u in usage])

### The same task with structured output

This cell repeats the Part 1 call without the tone sentence and prints the stop reason with the parsed object. Requires Exercise 1. Expect `end_turn` and a valid result.

In [ ]:
# Same call as in Part 1, without the tone sentence: the API constrains the reply to the schema.
r = client.messages.parse(
    model=MODEL, max_tokens=1000, output_format=Review,
    system=REVIEW_PROMPT,
    messages=[{"role": "user", "content": POST}],
)
print(r.stop_reason, r.parsed_output)

### Check your work

This cell checks Exercises 4 and 5 offline and prints each incomplete exercise or a pass message.

In [ ]:
# Check for Part 2; no model call is made.
stops = ["max_tokens", "refusal", "tool_use", "stop_sequence"]
done = [
    check(4, lambda: can_parse("end_turn") and not any(can_parse(s) for s in stops),
          "can_parse accepts only the stop reason of a normally completed reply and rejects every other stop reason, including \"max_tokens\", \"refusal\", \"tool_use\", and \"stop_sequence\"."),
    check(5, lambda: "sentiment: bad" in retry_message("sentiment: bad")
          and "zzz: q" in retry_message("zzz: q"),
          "retry_message includes the error text it is given."),
    check(5, lambda: "json" in retry_message("x").lower(),
          "retry_message asks for JSON only (the word JSON must appear)."),
]
print("Part 2: all checks passed." if all(done)
      else "Complete the open exercises, then run this cell again.")

## Part 3: Validate formats (optional)

**Exercise 6 (optional).** Define `Profile` with `platform` (x, instagram, or facebook), `handle` (an @ sign followed by 1 to 15 letters, digits, or underscores), and `link` (starts with http:// or https:// and contains no spaces). The whole value must match each format. Expect no output.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

# TODO 6: Define platform, handle, and link with the formats in Exercise 6.
# Hint 1: Literal restricts a choice; a pattern restricts the format of a string.
# Hint 2: The Field function accepts a pattern argument for strings; anchor it with ^ and $.
class Profile(BaseModel):
    # YOUR CODE HERE (the three fields are platform, handle, and link)
    pass

This cell validates one valid and one invalid profile. Requires Exercise 6. Expect `valid:` for the first and a list of three messages for the second, one for each of `platform`, `handle`, and `link`.

In [ ]:
samples = [dict(platform="x", handle="@techcorp", link="https://techcorp.example/app"),
           dict(platform="tiktok", handle="techcorp", link="techcorp.example")]
for c in samples:
    try:
        print("valid:", Profile(**c))
    except ValidationError as e:
        print("rejected:", [x["msg"] for x in e.errors()])

This cell checks Exercise 6 offline, including accepted and rejected platforms, handles with extra characters or a hyphen, a missing @ sign, a handle that is too long, and a link without a scheme. Expect a pass message or the open exercise.

In [ ]:
# Check for Part 3; no model call is made.
def profile_ok(**kw):
    """Returns True when Profile accepts the given field values."""
    good = dict(platform="x", handle="@techcorp", link="https://techcorp.example/app")
    try:
        Profile(**(good | kw))
        return True
    except Exception:
        return False

done = [
    check(6, lambda: profile_ok() and profile_ok(platform="instagram") and profile_ok(platform="facebook")
          and not profile_ok(platform="tiktok"),
          "platform accepts x, instagram, and facebook and rejects any other value."),
    check(6, lambda: profile_ok(handle="@a") and profile_ok(handle="@Tech_Corp_99")
          and profile_ok(handle="@" + "a" * 15)
          and not any(profile_ok(handle=h) for h in
                      ["techcorp", "x@techcorp", "@", "@techcorp!", "@tech-corp", "@" + "a" * 16]),
          "handle is an @ sign followed by 1 to 15 letters, digits, or underscores, and nothing else."),
    check(6, lambda: profile_ok(link="http://techcorp.example")
          and not any(profile_ok(link=u) for u in
                      ["techcorp.example", "ftp://techcorp.example",
                       "see https://techcorp.example", "https://techcorp.example now"]),
          "link starts with http:// or https:// and contains no spaces, and the whole value must match."),
]
print("Part 3: all checks passed." if all(done)
      else "Complete the open exercise, then run this cell again.")